# Reader request comparison — CPU, two cells

Use a NEW private Kaggle notebook; Internet ON; Accelerator None; enable the existing GEMINI_API_KEY Secret. No dataset attachment, GPU or model weight download. Cell 1 installs a small isolated SDK environment matching the audited run and checks Secret access. Cell 2 makes at most FOUR live requests, no retry, starts at least 61 seconds apart: router SDK/REST, critique SDK/REST. Payloads are serialized by the installed SDK and matched byte-for-byte. Stops early on HTTP 400/401/403/404/429. Frozen router features come from QASPER page 4; critique input uses a historical extraction from development evidence. This checks API contracts, not extraction quality or independent E2E. Do not rerun cell 2 or change keys/models within this run. Keep the Output ZIP even when the probe fails. Do not run Reader A/B based only on this contract result.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from uuid import uuid4
import base64, hashlib, json, os, shutil, subprocess, sys, time, zipfile, zlib

out = Path('/kaggle/working') / ('reader-endpoint-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8])
out.mkdir(exist_ok=False)
code_dir = out / 'code'
code_dir.mkdir()
entries = [{'path': 'scripts/kaggle_reader_endpoint_probe.py',
  'payload': 'eJy1Wm1z2zYS/u5fgct9INlKjO2kmas7uhmdrbaaxi9ny53ruR4MTEISaopkCNKR4vF/v90FQFIyJbuZHicTi3hZLBb78uyCb968Oc4WuSgkK6RI2KUUsSzg96dK6lKzcl5k1WzOrk5+YSKN2eXoavIDm2ZVwURZykVe6h5LM5hQFkpqlhXsp4vr8M2bN3tqkWdFyUQxA/Jauve50PNE3dWvZZmHUaJkWrqmP3SWut+Zdr+KmkKpFnJvWmQLFotS4huzPe69R2O+ZKkdZ7ph1XAhSwGjhJvxIAutYDkalosSWXN9F/C61+ZzaYbNsmyW1GvOZCrUnukpC6FSLqCpPOSFEWXNPr7xPEtUtAKqU8Z5LqJ7GMz50R6Dh0iE0ALU+UwuVKp4XmR3sqERi6h0XbIosoIvpNZAY08mWrbIfCWVvb1YTh2vkdBS+yrNq1IHhjYcc1WkzKcXfHwP9KOUhddb32F4Sc0nMlIo3149YXPY+fVkdMmvfruajE75xeX56cWEfcu839Pf0xMRgUSrdMbmsiqULlXEokqglKEtEin7VK0knLpK50e/p16zxrdMl4Xl/MZyyKdSAPNSe7dB0GttICpUqUDdn23h2HZcSl0l5dYtHF+OJ+N/X4+6NzFBtRAluwdGPxHzKpqzZQVNuINYpHM2U2IX/45BrqvFQhSrZgOBPTBjPzzLS5C19lFAGs96cAYm0GPzLLvX9NseI9ikZgP2+ERvoIv1DKY0mHPJcPBRzRCOv/HqQd4tTK7fHA1aZdd8+YBM0jCi8OhZN+MdsRtqvn1qa9mjJ3LFrYXCGO/h4A7MF87JQ/OGU4XGd/vwQAs6oJWTALTDZOuf4OXgqXV68HhWYMgXdOOfJydKkaOWcMuav8himfSYjuZyIXoMLAloWjGCl7vWEnwk2FaqS5EkMiZPWWZMy0KJRH0x3XIJJsfusnjVY59VOQdfyrLpNFGpbARJXhPpWh5ikNHNLbUgZ3Pwv4ksfMtZ0IjXjQ9Fnss09t2QsCqSEJ1azzn0MMrSEnYeBPVkK2zybyGoeg7yk/4hChUd8eDxyYw1IgOWyOGFx/Tq4wndy9XAs7vpT9USeelDIxwUHF4Je1eDHwX4p/VDWHtweXd6gw11NqydZtH9xInKt8II7Ebg7FvioOkhnZwOgV1ZQFzgduvmRAf2XG2jHpiDXecQOqdqNkA9NWLhC9A7Xq5yieoI0gYHIJDJtygqj9TQjjQKA8Os5nRu3Wtvm5T2Tyv2kz2eqUpB/55LIUoyOM7AGWkiU9+pS8D+NmAHGLJdy83+LfzDZu+tMba3RopvPfBJ9BP92pGT6bGRntesCgEQLOJXkVRyhJHF965TucxlVFrTsIpY2weJzwvadt/ixhqlFlNpIpVPTgTsUKySTMQ98KsrawnUD/ppu+DgS9+jRq8eH6AMICKhtaaR9GsysYrKgGEQbXlFdGStwUTLDm127Ja102BsWaFzpXbDhGnz6kOwYxQEUm989uvw4/iEDy9/uj4dnU1Qia7PhteTn+FlfDycjE6w6WJ0eTq+uhqfn/GT0dnYNJ6dT/iP59dn8NKhXh5gtfPry+MRH/3n5+H1laU0PoOAezb8aBf6dTj+OPzXxxG+noyGJx/HZzjheDQ6gfHtfaLgb8gl2/2gDzc/91oD1gAFN2BDxjR4B/LwW+KyTV5gTteBkio1OMbGxR4DO4EfPFZwKN/0mLVomNID0ULEfxDJ4MOB3YI9zkKG0ypJFqKM5n7h3Qz7/xX9L/v973nYv3086B3s7z95llaApoGTgOQuDR8yWEnF1j4QJQ8vxjiJGVj9qVKw8+b4HXNoZx8OdlGeQOiIlZilGYGfHLwLzJSaYsqHg76W4KLi2qYcYbtUI59wcQ//+wjz0ddNigpQgVwCpOLZvfHN1skD1EDDtNE5yopYxvxOTrMmJKLfMRQ82jG8WtF7Or5vxWv7y/cMYO5T4PCC57rqLcRSLaoFT9TD2jLvgabzdjy3kBTdHvlFGsatx+RGFpo8Z8cSbriTUWu8awKiSJ/jpgvEJBY5voVtkW83bwUiFhxrYZnrrt9pwG0HDzrKTOS4NMy8ddGCwlCBEKF12lmKUP6LBJAIppVk+QLjr1H+H0gv5ZImgYwRWiaqXKHGUiRgo8NR2OEWPDOf67k4/O4D8GLTsdA0+BjHwrha5Lo2M5lqBERCR8rG8SCUaQRH7gdBOJfLWM0QKQXGAfqN2rG3RiqoUSFFyCD8jO8cQEHZXsuNQsuNYZuDwwAXhkVUOht4VTnt/8Nq9QZ9w+Vu6tt38or1jBXA7ESA+oC/K0qLzHqEcmkQYboNzpweE29gkyEcPhjD0utYiQnNkmzW+AJdZjksQkzWjWCFrMgSuQlHMZJsz9ncY4AgYlAg/DqYuzZ/fXs+MoJgoO+gpTkCdwZ3qxIYwdXWqeAe7iDvBZxqAqAzLjSaDY7xoahi3BEsCAprBOBZGtBgf3W4I0JXNlgd0WHtQKCes0FIyzWOtwpSe45tJkN7bNvBrkUcMRIP0EI8RgRano5w3TMaCN7aGtiZaLWfz0Kh4MC3+vsb2vttE4P6VCoBsAzEslRFfhB0UoPlkWD3UvgQGZ1Imfs48DmVxnQ2LGmTgW4dgDx4LuGkYy5KXpURIQpX8AnT7LPvaj4h9Aah0hnoGgT6DoJ1LgNnnkrjQQeNNa9tq51ZbAjE6fFgQKHP2y4cbZaz2VzXgxBnVoki7sjvtiyPRHcPwsegissqRfFYXHEK4RwwBCahkIC2YflncERWD2vIsmtTLuXczC9f4r2doc4h3lHOYdMKAAGJSEEYMxka6ACgUwPphYfxbTO5xan0t9Vn2cAutK7XSqmNvVAqiEdc6gCheToFRGPqa6rUdQLflcp0PbszaMKtr86Xu56dOTSVgEjBthg4ScHhkcFfl0Q/k8L/O6nefP7CJLvroeI2ujHDk5EYp4wABOY73kMzbjudCnMeUg4IWGVRTwTuqIu70rVncMfLlOiv5QeBkE/Zr+ltRY+1rHfzacrKXc+a+2xV8sOfJ5OLq+O613+FaZuaPRzL4N3+DhuqaYYOungX51cTSvEdvhlQoe3PGM+c0JMGrVz2ka8+sEUlrCOTT3q20tGfbFPTHRrStqqGfThn19ERodxjQ18byZhM2mpVK//ueuiqY0CFvBCdmG70EQGjf7B/+J59w/DPDrdA/nptQXSsh/v7u/3q86oNsmOT+l0Tu0LWRZE9KLxPQdUyxZVdrhaycLI/bUCCMJWFptmj5H5HPAZzLTUBZTflZv/WUrFVrx4YTWCaaDQ03NySjbWWJ+PasQ7mKrCM54V/ZCrFFL00JLEDnR9wipgZOxAwG75sNaM1eo7XdICgv9ZRcdQRHxd92bE0AqWW08YxPW4xA4fg1iA2KjIWErbBq43yG631rPy2jcct/sy40lQsSDN8ODkKWZMMgNxxVtGp2jZeYiNEPGzdBeqbp61h6ySb9q8ia493g6hr/TqSWSmSDXrYtE5s53GSMM1VTgQSTY6wEFZJ0ldqgIQyFfd4O9IW/CsdM15Lgbf1Dc0jF89Q74j4JgKnDNLYG60aBBjlINGhkpyh8s8B2+9WjL+zXyB1ATycrCCGVGkMphLb20s2VTKJqejCHBKKmVzmEMsoBBD0FJ8p+IS7JOYoktD8R0hpMkDRIPIM81ZjoWGrsUdp6RQcH1hAa0Td1r2ZbfKknJ1kZS5tSVqvp/DoKc1FFMm8FHdJi5+1ZqwDVjNMhTFXo2DXjNzs6fAXcomU2Ij+oHAhJYG2bpNfKzoThqS7SdQbmBOEnFSO887JWMNqIS6YQPKO5U6I5fQSBzoV2+6PtgRwnPwXpJXPIuz6XmJ4U4k2jnlL4H12cbTBjc1aXiw34NNx59RFsUGNr6PagKZdlJ0qJCIHTeMLi5TQln1/s8bA+q4eERD+wSzgXTddWwB0mS4t0z0yyWam+tUuPtqD6aiimk8EtqAYpDVNKj3v2C3IsFOtsJ72Hnfyfv8A/3uH/72H/w6/3xIxbZlxaxS+A5R4v9ZD11dZ/pxcMxSDOwGNR6vJRww0zBfpyi9uTBnvFpWbvBH6Zmh15TzqcAZg+zZQwwv+ioqkpiJK5xbUZVNTb6y/V2k+/AiCF8qS7rME/A6EapGblUZX1jNR05T13PpNsZJvlhiNpDYDtocCzkGFMwjDNo0v+KcKorMxdkyAYUizLtJydepNYvZmgM8Q5Nl1zWchNVuIdzLrpzsLD3AEsLlKx1vKqB7YFaVFVyrFT6OyO7ynotCo6TMX6/aZaH1pYuopGMQga6vgPTOXGhqjiNJzmAgR391qPL/rCL3OCwf7kczuGwFzlF99I0DX1IaGvZ1cCADv1sgo1OGVsPv0LRwWswpvby6ox1q0GRaKOMbvUKjf9/p9U7vHdBiCzOCi/noD7xDp5m7nbCOLPsjiRQr2IyBLiP7QBzF+sK7tzcVrK43EgaG9e8FM0kj4mbxAhjSydVW7zXxNRSnToUwfVGESZN/7aXQ6Phvz0/OTEd1Ym7vj/rvwQ3+aCD23l8Tb5g0vxvyX0W8Ox+YFxOo/rQdrp76PHhDdmZl6s8W2b0MCnCBM+1XBwZ796M/gEQrunKPacG4jvEl7r1YAjxajpSp9o1TB3v8Afvdj6g==',
  'sha256': 'bc448860fb79eb761cc7a50c7123592562c15e51d6b1bbf99b6c6b4bc37f37a2'},
 {'path': 'scripts/kaggle_gemini_probe.py',
  'payload': 'eJytWety2kgW/u+n6FFqa6QEZCfZpGo9QzwEFIexDS7A2ZnFrEqgBhQLSdHFl2D22fc73S2QEPbMbg1ViaTu0+d+bWuaNlg6vs9O+dILPObcOp7vTDzfSx9YFIcTzvhywl2Xu8wLWLrgrM8dl8csCFM+CcObn/DGYp7GHk9MTdMOvGUUxilbpGlkTn2PB2m+9DUJg/w95vlb6i35wSwOl8x1Uk5fTO3k3zUB8z0MFFwW+743MSMnTjaw3zLwU5MPO/Kz5ODgwOUz0HGdaWrPhXg2j+Mwtpc8SZw519Wzxm74g3F8wPCDAGecR0JQiH/rQdQfE8bvI98JnNQLA3a38HwOvEvsBnM2BQXI6Dl+wpzAZdNwuQSUJxZnHo+lVgi5NyOtMS/xgiR1gmmBgySNFQf0gzqzOGDq2AuonIRgEz4LY87SOAum4CWY41zIHGIfbIRJQvwQ52GWRhngwyxwnRibTkCEfe7cmDknBS5wXnIgBMDXlhMQZLeOn3GyfgJFc1dfCXihaXXUmfGGphlF9dOOsRaqbfg8qEGmW+iCN4Zxxgui0k9pgTUAZCbZRMeDJ1Mn4rqgDcTaqG+1m62h1R5rNbZR28x35kkD4J3Tbq9vtZoDyxCoIydNeRwkwKlv1ao1O9+dUbP+L6f+/aj+D7s+fqXVCtvks8nJ8eHh6N/Xyc8frrUfdwAKZ81Xf8PxX7Yr5i6264l+cgwn+sqnafIYxnMn8L4LH0oeZ6EPz0oeswT/G4dFvMe/PIfocYrj2ZLHj850CvumxnXyEtue+xhkiNTYOMHC6LgxFk8SAcKQIOL9L8BLj6dVKHBeu6vXtbfra9NY4X/1MSnB6Sc//zC6vjseG4AfAZNTn41XR7W/r4+N1ZvaurSknyjYHVJA/b62i1lwMOFOTLJE3shm9fEJ3JAEg9/I7zS84cGjk6WLMFZGMQrywfy1nyqSSaTXyauiuf7z6hA6aLxUsNL9KGyUC1Lg5N64dfuKyyuQ/93Tt5g0pplfQy/Is4qZREjjuiHhVE5Re6Pj10dHR2P2iumaaZoaZQSEaX7UYB8YATDuI8UitFU2nS749GaTTAM3ArlUX4Yu9xuaXK6/Nd/XwW2y0FSYxzzJ/BQMrjRxnru2k9pZOtWONyneDMI7Pc/yJvYM00tCqHHpQIJaKV0gSwuKOC6e0FnOiq3ql8+x+QkpGZrTYv4t40maYGk0XhczMdQ5y3wfNKYLHdENm8Kywqz1MbwWKliTEYhKKT2TQGYWEfe59F6AXOW5AJdFJn2IkBQ7cvWCYLrOkmN7Bg6zmNtIvnNAiH3gsaVMxm4RkMTEKuX5ButSHaRPgYDsLkugfePM5z7wcpQkSCtg0riYzKl4lqHy8nmFPDSQSy1Zs/NDGyrYJDblQVtWdm0LVlwGeAWhblRRAqNbZnuLUApbQmvO8SpXdO3Uuuh0O3bzsmOfWb9re7BPMs93bWX9LeLIefBDx5X+GAYpEAvXWGloKPLXlN+neNP60gi/DnpdduelCxbegCc0LSGVYW5q6/F6vOue0kfnPOCxyCytMJh5c6BbaUvnvieK85ASEBF79/qN8NEkQgLmFwiBITyHaDsR4ncqMBxS66TtpSNo5ccHCLClIyilCkvv469Wawi/05DpIx6n6NMEQHhTgvvY651bza62Xj9JZj/lb5mHFojURijH6/W61D0Ic1AiQ6p10Stks5l3X0Nv4j5QahzpGrYcRIBDPJ5aglX8IzevBD5DutrqlSAvewNx4lgt85Y0KdZIZ6abLaNEVzY3jHG580BPEsjMJNiEEOK5LgHJCBxt88jYhGmQcnRx3CgB4zw1ScAp0hpawTANA2+ql8HgeAGqLvWTjWKrbH4eDi8Hrc3uVtxbjg50nlFen4chIga1LTHRbWqyPUbP13h7VKZSiv4qaVNJpOfG0Q5vX09gjUORi5JDDfVBvOJZNFyD/vtTbrIQw0LSWGn3dWK7Dq7rCG1oWrSOmjJX/UmnXxsVOrm3Q3UFYZAc8g29ekaYaiQ6PMq8aQYziuorT5hyrXKM/BJgwpXIgxJ9c4KSl/7+3bu3740yuZkXYKJ6XvNTP3yOTdg6SlAnl4pLauR1fdejWD13N4O9FPW6xt6WcaLQ7ZX8hwZ7c3RUZVHULlAkuUmjuiZW4GOrPYaQ6AAugCS8IvGkaHCAogEUCvCZvwWFjr340zrdL83zTttu9k+vLqyuCPurbvNq+BkfnVYTLZPICFb/ojMYdHpdu211O3Kx2xvan3pX3fYTSVTrW4PeVb9l2dZvn5tXA4Wr0x1a/W7zXJH60uycNz+eW/TZtprt806XDrQsqw34CmJDtU9X3bNu75/dKsALJobNGzQ/VOgdeBmcxOUJhmoal/ARPTCLlNsJZhj1ZGvO8oxpVjAKVLZEJYYfbWD1v3QgV7szINaFWB8752D9tLSmCqmttPyUnnKwHO3H817rrIyBUpjdtz5Z/b7V3wfQubSb7TY0PtjuPk+t2W33e2T6y8u9CHuD3a0rMGhf9ntU/DZ+sJ9Iq9cdwKH69uBqcGl12xLBZjVXSNWjXZjB83f8Xy0CxWj8dHyLFlGZScaBmq3lcSo2tIctqqFykUJDYd8rSHmol5A15nrTVA72ckVy+Yso+wZroE+i152acig/zTiamhv3q/ov/XYxK8YN4rbkjWtjL4dJjflekqpYGY0rRLYDTkHLalH0Cn+gZQVqy5sgdCoy7//Ja6EK7rIMpQscoQsoIBK2ZE4iblxAkXSRD2V+eMdj3aim3gLjOcsLjDSC3Y2n963LXh/JyUavdrYv55Rnk/KsoVqxfFHapSErg0zzVXEngLrZLSmqxQZfm95tTylRslD/YIsRi0tX39aWgCYi6YS7Hcc+vdPIth/rfm3+H7qQeIkve+klAv8epexXDDnwnqIPpxADXknyzapIFKv12BgdVV0fnYSXLKjNycHlabneV5G2T1XqJBVTbTDsXVKgXDR/s4e9M6s7oK9B85M1/J3e+larM2wOUTDpq4dy2tee90+J3d7kqIaiVzlDYxS5r7qYoAFLSiDmKxG86q4EMyiY3ZFzuunl0X3IJTmj7U+uVTWQxxSILsJsvkg1o3pU3CW75U6PeHwytyDesylmQ04TpuwIbXkBQOooZAiJuZiH5YpkCRMT0mLC6Fb0iVge7bvdICJ/zMlTQfQc4r8slDbMkBVjyrbPRFSmUvw2QsTKxXY03Nd/Kg0IUKGSFUXuscQ2ondZP+kN7rVXMp3m4mUkB/IWtVfkl9v4LK8rF9pdDVPHLyztd00on0qtLrgTQhJfhrA/jCB8o8j5hwY7kpMov5/yKGWWeNDM6NCfJKZbU71gfedOgdG+KiDJIQY1zPts6TzQ9IF6G5T+YAHd5D1l8Q8WOzddeSUVPQOpWYgBaoZpy1xpV06UXCTv9OfSxysD0s7FVvnm678WMNb/',
  'sha256': 'd0fc2583ad529e37cec4d2287e721c9f1314e26673dffad79147d31ff7cc9198'},
 {'path': 'train_agent2_reader/reader_policy.py',
  'payload': 'eJzVW21T47iy/p5foeutW+XMJuZl5uy9m7M5cxhgGc4wwELYOntZyiVsJdbi2B6/AFmG/367W5ItO4ZhtqZu1c0HSPTSanU/3Wq1JMdxBrngocj9LI1lsPKy1WC69hmc5ukyK5m7G3GZjNP5eBal1SIqh+x7dh5EYslZWLFYioq5p6uQJ6UMhiyIUrbNbrlkZZ6yoOKDgzRdxIIdiKVMJNs5PcSaZME0D3whkhJYmAwY2/LY0dFHdpZWpcgZY2P2qVqJkoUyiWAw6FSmacwSnjI35itoFi9vX7MNlga5P+dxfM2DG7YBlODD49IvxGIJ5Hkp00TxFvCAzc4uTo4P2PuTU/bhPX4rI2GPFETQCP4kilAQSXbNYfBIVLksYJosTBO2kDxh7r1YdmfCQsGuJRC7gY7Aq2YHa9kilVCdsJCvYFT4cwuCqqzBhx4031Zy2M1lKT9VAuQA4o1wRGSrZHGFAryBLp8qDuKUQcTuK16OLEJ61CAQoEQcKRdlDuMBAyTEG5ikNxic731gezsgg72L44MJ8IfKGgOvoEEXKz+eHKIamVLjSHGNAoOpDJrmIgch3wroFXKYPUwyy0XASxEOPbZLxeWEeMpkxmRSlKAuNr5ojTgY/BskX0ZVMMH2KMWELZHjnGZ8sP/x8PjQBwz5H/Z/G7EA0AdCLQTKMMQWYRqoUcR9luZlp8fU8TzPGQyOLi7Yb+zXfTbbP4YJ7u0fTZSECw6SXKahiA1eabohshCJFP7AN1S8x97B0ANNnkiwa9RrhQpOA2Soyz3CAiAD/0JOrEo9FLQZJKhXMKmbCJsG0K4AoQdQD9q/kYCyMue6fYSEA+AGJBaVZVZMNja49JQkvVDc0ogB4Q/wsvIGDtj8QC5JJGlhvi15GZnvfxRpYr7H6WIhk4X5mfMkTJf1L2G+lREC32pYymVdWVUyHMzBhTBwFDL2qlLGBdOVGc8LAQoWfpn6+J96UutylQFF0/AIzG0Ef8Ed8LieAc75XjXPtOMxHd7xQnxEIY3Yz1LEmgUlGdNGQc2q8BTcdbXI8zQvBm3dTkFsnkhuJbguaF66jl3tjJizILyMX3s/jOcxLyJnOBj4GqBTdpwmYuDnAqy5KP04BTc1bQToHUGBOxz40LGsW/G6IyoEHOLUaAY5OKIy1zHOJ5Mw4tn+7Oy3nXdH+/772ezU3wXmzqEbzPVPkRTA9sObzf8esTfbP47Y3zY38c82/nmNf948AsuDAHgo2BlR3cnkuyqEwfbvAyFCEbozUBS43X2U0VBZGmBrBu4AWJMBj8nBB2jb4j7iVQEOgMmyANTzMJaJ+DtLUqgCMAdYVcJXJgvtp0ToEVIHoZgzIwa1SLlDNv4HOLZAexGZACZuYTyYXZzy0n1CP2f7v1zsn8/8w+PZ/tmvO0f++f7uyfHeOaps0xkOidg1TfJLpHZ3jo78dxd7B/szm8r2m5qOnMPslF15spgDHkrhGk6HLM0btn9im2oi+Mm5BB/2K48rQZL9IvdsCZIFh8PUGAwsFEZOErEgL+w8w46aKzGjp/3T9EW89Ey/j40sLaTFA6w6VZ6wB2fJ7wHRpYCYonAmDBDnGA2XClSwWgdpElLtprc5qlkilCGmfMWx1VAVWLSMgK02pqhDEGOFdD73kXXZ6rAFgzf1yHhT92aNsSK8sSe29ajxi4wq+1fYdcjLeLtU5CiBL+L0GsCg2xmlGbcBdoHm3+im8Sc2LbfFDzgC/0aspk+AWK+EzrA9C3SpfpphnFRMH1pVNEmtIlKNv4meg33HljKOpRYNy8BBaSWMCHYYI9xFKfjd072f1ylSOGKGBLrrg1IzW7I0aCt0ZeldUtBIICeIayCo0HGOAsYazcf2tK2fLbgahShN+opXPgccueI+IHWSq5joTkUGkxCkFwg2S2pFoFQV4CZQkWqIWzQtq6lpBO0jmlsBzR8eh0ptZzj0eAeHbqxakViDh2YebIeKoGNTZ9RkfByRGOpQiWJEd7bKlMmPLPMfNiRa9No0exd0PYaH3yHcW2ZoBxQkUJE7rIk9y8GIndyKHLi+6zLUM2f9E2zWRRdiWByi1Np+sK4RMWgO+ytlF3wufIoB/BA0BFFLrfJm9YFVaicGjgD+EH5CtBXAsLCcfaoAgmwJ/UAIHNY6Aazj0ghlRQFbgAL9Ls+vJSyAgFLTkpY9JJzxFagnXEOSZsUACQ2hFapve9vbuPSSiBpA/uv85NheCmRBkXcSCFePNKJZDddA9PCosEH0poYvhUkqBFZ04fCJEYTS3pP0qQhFLmFHVMq5BOtSkGlar1OlFiMUOq1hWJsLb17FMeg3iNzcudwZ/w8f/7k5/tH3Jhvjq4et0dYPm4/AcJe6xQ/FWRawoRiBfdlxowlfgvTnODB8g6WFrXlW0NbByckBxGBWyfuf/dnJh/1jZ3hlz40nK1cNRmun/gpUlYXjOE2Z5oqmDVMuBM9hvmsuLnd2Dv/kn6O5/3kRXWZpVeRX/ucsT/8QQVlsfK4KcDEbn//5mbYObydfIRddRs0HxvVVMS5IGi+gpLJCwVk6JRwo5KhqK1xSBTY6kN6laXgFlNRXDRayAii0aDamcXml6KLYVCmKTddfTrY3r3BIC0u6asTQjrUnuLx6Dn2qxxqo8QPupJRJ1UjrRiZoyKqL4vWfsL0xkZEeQLWawhILVXoPB2t44QXpckPvT/Is8MiyD5N56rSHlVCE8oc1f2JL3RoWqoYKtPBNYRZWUdjwITJha8dl4gwfO9pXelCOEMcgXTzcTNitogSoQVJY5YE/XRYuOdlbXJZQaAichqaIv2Kqv6AX/Rm4B5/ame2tTGPK5hQdyTYVFhBq1cA4CVrzVasY51H3I6urifShpal9EjBPA6fu3IudJzFUEwQJ92m5JvuUomlF+igwR4Tapp+HYVfdJFodl7RJqh60GjvDvnmiLrXbphEh2h6pGGNIPm0T9xeKNnzZ2nz1ausHqmgvx30uyJ78pc0IQlE5oW5TwtkTqwUO2l0qYJFQ68NTy8Nf4CGE4CYpNEb7pLlXN1CRXp9YbZ9TN38WOxaH1gAdq1WgoDQSwiFOA2KuR7Vf+gCP7i2btDx9w6mC4xDQ2OsOzIcM0+NZJpLQ7XEuMKWnncuw12ORAHzLHaAEaJy/5owoAO/xu2GVK7/R9kO0VdgTMV91ZNrRqe79ZWj++Oi+nfzuNT+HbxE2hkAPGIwk1K4lRF7qDSwKA4Nj1cZiudNuxDaH2oxrXi8n462rYWufpMiYbZJYytKFiBc9QJqYTbHm8Du2lyqnmARxFQqW0fkCYLqOVq/TUAoouDg7gr96M4TRjtofoLBNGO0RTZUX83ANcp3/RK4xjemF1TIrDCNFmpe4IS6msxysu448GgZtUDXSbOpdNDpFbjg0c9Upb+Gjy8ZN+CtUJbBmT32KJEfs1aubO54viiZhdgK7VbVPhQ2sgCC5iDjuHzIeYPYTbQB3DitM+iYFpSU1yRFtUJqNgwrDMAFXrJIggjgVAj6GiXuI8wqGacB/syzimCNXe/hCbc9hwJDd8TgeB5SODBCXsfZVNVGTtgNtkQBAUuLO8FJo3FJgyaCrqHVZ/F3tyukg5y7Nb0Su9qSKB2iHUQdSC1CjMFOcM2d0LACCSCiPtZGIEjtTQtEzwmtlTjoZU7WJUrmB6VoK0YSoOaYep2orugSJlWkiA11dz3hat/xeU7zsTUKpmIJqJFLFzLeHf964Qy8S96q6Sdw0WSFjSZhbKJvx1BxA3hJU6dMRgM4D1zsmzYIwaVmjEMsbKGt8cEhtzoRZiWK/KNMM3b9mGmr1NygjEEOJAnPXueilY8IUoi/176tRnakBhisa0PC0ToOSTTrCn7RnCnQAHuCCkQXH6KKeqLNOjJI+PjivrBChv0SKOXAQum5XvWxsJDxkryASwSzW6+Fjy1fZKQmQ+lNpcEdVNMnuGjU1p60gv63NXn+jB/W0vls9OslZaDaoNzq1E0vwoGYh3K1RDddWzvUKcLxlAeSOy9L/gjEogKKpoluaNtMct7rbU22aY17ZbNFbRx8eDz5VMheu9gbTuk9nNXsC5jD6VsPfWloqSe+emw9+lJ9VVueunbo09v5kVvmqi8PmgwncLkVL42rTADy2OYKYZCGvY6F4wiUamowaRke1o1gLK+yu/2h0tB4YvEicfUSR3XVyJOEiFiJz7dbj9ckZSbwAbXr4bo9vM6+e87XOQHVz2JaAXXeQ1UZxLiBiLOw8pjnIWBZah2CLAB13KRO3C6nuocfVmmvr/VgW2GHdOLVhww9QnkuyW4xglMtWMZ+qUduPbvtLxz4NUDsI6wCgmeSoJ4vfStg/WgmAW7X+9S9IutNfW5Nesio169GoPsxQi4U1nR7C5OXsRaULzLYjtBeWPnK6We9ytU65b7V62u1ZpxAqwPBICqjwbqj66sGEpChwBYWJ1v7jWlp+v46+eYFl3bRXuH7wgYWtQw/z6YY1aj3UWQT8QRtGsBnlJlvZR/y8wLcTJDk4I/Ttrp2PRtbU6b5nUuVq66W4SFjf8fkzO2M8z+2Qp6sJ3sxE7WqMjqevz8Wn9eIN+9D+Nbu9HKoTSfQtlkPpO6d81pl0O3YPQK8Ab9uwaWFPO9L253t9RcSrEgkhyVIdvWx5m92Z26cqGO/2H7Wsa1MdvWmfunYaN2oTfn5TO1Sri4FIffjTMya6K8Tb9zSsltbI5qjNqg5cEXY9kavKCd7ZKxkCyG14UZspPbTV7EnRE/NOAuFrTcQxqce65PnONRipYwNNVUtEOnkMct9eleEpn2tF8tNOHK90gnY9JeNGTXm+j8cmvj8ywprq/8/gtaXcaevXc71qDzvV/hWF/+Si+YTDNp/uJqNN8+toWfhpKNkwfxGVRNyXfm36hozxEA0JSnUYYE6NSmvcd+KjJzI43ejMJriO9BcHaE/S/Y+a0Z7sFm6Bnj94+cIi8R2lR/DE1Nuke4KYKKlXT5N2uq7K+hpDUQWYpphXMSVU9JGUV1Ps3aybyicNRiWHjCE4epDaUfxkbSrJGl+wG9YjfjPsfzXuGyl/EUvr/vBFmz97Y2pykKS5wWDwHRt/uw9QO999v/9xx77ePG4fuyOQopSuOpd5tRIJ3hfF+5t0ic50A0q3PK0Z9Qt9exoUm+krnXi581aoO84Br5BaQKf3IyIMZRne9wVKeOM35OrGBaygCRUY0p66iOF9Y0noO4GUzdsTgcQUv1tftdSb9iACQ0p8vFw8MVc2L53mmjYG9/Y9bfzdvaNNmw26uNmca4eiCHJJAejU2aXbsRXes8Ur2ildp72vWKxSpQu8a8QSvlJJQpNhw7NO2EbjBiJ/foD2cbpzIPEie4T3q5MFyHoBA7qBuRBfmgvxsBiBKlLFUKBZBC5ghm16IV3Xxtu7PAAtBqjc9sVyDl0JA1nTdWhNhoL0UEDAOVHp+fX5LMSUArBYTOkmW0uAe4BUieOv6FZ3c198hFeWoR9dT4d+RoLmRqi5jH5GKf817cvCV3c4MeaYsGu8Yv68Kj+sXV/HO8147du64w5fM5ZEPFEap9vRtm5lUYBLm9BF4UvQbi9+5hw4BtwFZZqvpnh0O+pnaa++A36rLmpnyAldt6bhXXq4kIiqPV11sqC9sU4yV4uFwEgKmiHtxiLqkR1FwHLiOmi9k2Xkty2lt4ltWr0N1qzLajaH+En/bmTWEsZ7FEaoL66TekoJ/grvwTtfb1gvNSV8o2E9xUBjaZD4rf376dnJx9NZzxOXbzzS2cnFbP/MP//tfLb/0dejTvEs4x1IIsZ7/SW7WQFqwBXgIwMUOj6syfC9AaAOHa+STSYzgWvlwDKbBN8c0OsYEDE9yAEv6LHjCJ8N3FZk6NdqIPJQW/oFzmvjqwZZBCOnZGklDbPVuFM8a/yS1yJvrP2WNxjUhGmBCtLJYNzj6yewQH6s0GFf0wkUrV35CHbhI3b8fuf4/ch6DXFweuGx3UgCFiKcIXoOfN+g2MTHEHiTHEwCAmiy4F93mDIQtndyzA4Od46ZK9HFQDgi8ZwIn/TgV6AHQLuH6d2IoYectmxvwmYQlQm8r85Ods889m5nxt5dnOwqkTc8qKdEmhOgr3hx5xK2dgVMYYPj6x6YA18y1EJXHiBnXEGIGZAxqASIlJFa6GqHSB4KqBGjlgeYMPLGbE+AjR4JnlMKHpaTOYBlXFZ4PRZVuJcGR3x1jHemd9/vfARHB6PMUEKnF0r9SYTzytDnwvoilXJ3d5qwRIvVnaHvG7GZLGPav50LcnVjFUF7nkezleAvk0ZWuvPp+4tdNts5ZW5C6iS1kEZIwbVCyBFD+RD1qeSMJbsntr5dFUMpsMAqBmpiJOzB4D1Hq1qBy0E9IRPX5Mh71vD6yUzjfiaDLY/NehBGQ70dbIOREfMjMy9cwczbMMWPW1tPkSqgw0q7YrTfjzaQ5vDt4LXH9kyzbcUifh9ZwUTKbCN9OxjMVMj4YX/GfrnYYbsXJ4fwByCoXAE+V1Lv5pB9DCWZiTvJDYzM27ecD4DMQtKLMBAU6hyb66cYu2eHs8NfLvZf5L1+gWAZLDOWMEMIGqocM2PfxHHVD+DwwZ1++WaTcsN03a/psHr4xcBiQLrqeDp89wc6u4UJ8qp+69j1dbvkEyvllcyTvA/7H1uDoRcE7a3QEf6qEM2I1SiF7tgD/BCOBQK5wTkpuyfAo6VbUZvyvdeoWbcJBocqlIqQpWYIGJMzZWCgH7CMmG/EMAT0SPHdEUjNcA42s2H2FJLc4Lvr9B6olMQLtr0NQU8gRNAJVilGQITVSE3jtsK8E9cQIt6/kQnuNtogSCupWtO3pII3PnQDsjZSfprYJlujDg0adQst0LDi1Yje7jWhJlmmMnEdEo7InTVvKW0DBeGRDYGhaiI6jAzT/zt7NU9PVNDgh2a7Vq/W/lxwvK0OUTMeC43Yq7UrK3Tbvb3dq6+u0P91YviQNayaHfyDE/HCRwn75DJxHc3xKp7Dbxc+aaa+KfBm20o/OHLJF8LnEFz6dO8IGsC25G/QM8OKOxmWEZT9sPmjKYqEBPxA2X/9+Pqxxae6Y4QZ2eZgz3noC8cef09+Txyr1V5PnBOQe+Daoi2Q/p48rIvksb0JrU9n1u8O1cPSAdfUUbpzWnppvjbConVvar9EbOo0bczLpq1O6pin88infqriL+VSUNoWj+h4lsVSXRHc+KNo7SDavRRUoUsbNU3zx1FLGK3Mjc5a2O+m9GbT1S8FgZRfVMslz1dfQG17m9pG7TqxddQWaZUHgrIYKIBuwiKplgq8dL7Zwi3iuvFIhFoANnMKvsxiQaaAnS7pDTKU4z/7nOabwbt/xV4D+Axf24I/X39GTug2Cy5ie11wfx3bRrX/L9HdRtfXoft/AbNZD8k=',
  'sha256': '1b0d51a272b6183d2244b08ac2b809602a6ee1eae4f2e135f394e27d8c1571ab'},
 {'path': 'train_agent2_reader/__init__.py',
  'payload': 'eJwDAAAAAAE=',
  'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'},
 {'path': 'inputs.json',
  'payload': 'eJy1U8uOnDAQvO9XWEiREmmH8RvYnJL9gUTaSxJFyNgNWAGbGDOzo9X+ewyDNPMDuSC6uqnuqm7eHhDKJtPWc6+okNkTyhoiGs6KsipLMByEYMYwQkG1jHFOiGp0S9qGllqwpuTSMIk5E9AIoQ2pWPa4caoOaucTIdvi4JcIoW5BxSXAnPC3BKdEr+Y6wmusB3WBkPAYFni85tSpq5vB6z+19ouLKUmLPWXHtYEKoOqgol0b4Rzvya352ZrYJ1hUIqeFxBhXhRSSivuiHmzXr8QlJ3lZYUy45GWSL1LR+za5DjbavwvU8zKOKlxuo89+CRrq6P2wGpcEJJHDeLpakArcMl7nX/USWt7J0t611oDTsE1eFVySsiA4WSxFwfbKWY3TAJs/K8WvDU04l5juTVL0fYE5WeBuyMtlglv04fb6TU0QPs6fsg34/V/9au3ruuvaB9tZtzr0RWuYIhiU9ASl15lRG/yIDJxg8NMILqKwOEQxlQQT+oIp57z6+Rk5H5GD8/2XPiDrDEyQHi4OFxTgZOGc6Lt0bc6slxT7/HqP+66WsK2qj3Gan45HpQflYu8H311yH7pjakxypxJ+GJV1OZMiT7/HlWOwGty8Lix7fj58/XHgOc4e3v8BvTLsvw==',
  'sha256': '6bb1e93800255885c4ad2710ebf28d1c2660e06a2b47f58f3a494711cda344f4'}]
for item in entries:
    relative = PurePosixPath(item['path'])
    assert not relative.is_absolute() and '..' not in relative.parts
    data = zlib.decompress(base64.b64decode(item['payload'], validate=True))
    assert hashlib.sha256(data).hexdigest() == item['sha256']
    target = code_dir / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open('xb') as handle: handle.write(data)
(out / 'code-manifest.json').write_text(json.dumps([{k: v for k, v in item.items() if k != 'payload'} for item in entries], indent=2))

def bundle_endpoint():
    items = [{'path': str(p.relative_to(out)), 'sha256': hashlib.sha256(p.read_bytes()).hexdigest(), 'size': p.stat().st_size}
             for p in sorted(out.rglob('*')) if p.is_file() and p.name != 'output-manifest.json']
    (out / 'output-manifest.json').write_text(json.dumps({'files': items}, indent=2))
    with zipfile.ZipFile(out.with_suffix('.zip'), 'w', zipfile.ZIP_DEFLATED) as archive:
        for p in sorted(out.rglob('*')):
            if p.is_file(): archive.write(p, str(p.relative_to(out)))
    print('ZIP:', out.with_suffix('.zip'))

venv = Path('/tmp') / out.name
python = str(venv / 'bin/python')
setup_status = {'state': 'running', 'stage': 'create_venv', 'base_python': sys.version.split()[0]}

def find_bootstrap_pip():
    # CPU images can expose pip on PATH rather than through the kernel interpreter.
    candidates = [[sys.executable, '-m', 'pip']]
    pip_binary = shutil.which('pip') or shutil.which('pip3')
    if pip_binary: candidates.append([pip_binary])
    for candidate in candidates:
        try:
            result = subprocess.run(candidate + ['--help'], capture_output=True, text=True, timeout=30)
        except (OSError, subprocess.SubprocessError):
            continue
        if result.returncode == 0 and '--python' in result.stdout:
            return candidate
    raise RuntimeError('No existing pip supporting --python was found. Preserve the setup ZIP.')

try:
    with (out / 'setup.stdout.log').open('x') as stdout, (out / 'setup.stderr.log').open('x') as stderr:
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(venv)],
                       stdout=stdout, stderr=stderr, check=True, timeout=120)
        setup_status['stage'] = 'find_bootstrap_pip'
        bootstrap_pip = find_bootstrap_pip()
        setup_status['stage'] = 'install_sdk'
        subprocess.run(bootstrap_pip + ['--python', python, 'install', '--disable-pip-version-check',
                        'pip==26.2.1', 'wrapt==2.1.2', 'google-genai==2.22.0', 'httpx==0.28.1', 'pydantic==2.13.5'],
                       stdout=stdout, stderr=stderr, check=True, timeout=300)
    setup_status['stage'] = 'freeze'
    with (out / 'freeze.stdout.log').open('x') as handle:
        subprocess.run([python, '-m', 'pip', 'freeze'], stdout=handle, check=True, timeout=30)
    # Compile the portable source without calling a model.
    setup_status['stage'] = 'compile_sources'
    subprocess.run([python, '-m', 'compileall', '-q', str(code_dir)], check=True, timeout=30)
    setup_status['stage'] = 'validate_sdk'
    sdk_check = """from importlib.metadata import version
from google import genai
import httpx, pydantic, wrapt
assert version('google-genai') == '2.22.0'
assert version('httpx') == '0.28.1'
assert version('pydantic') == '2.13.5'
assert version('wrapt') == '2.1.2'
"""
    sdk_result = subprocess.run([python, '-c', sdk_check], capture_output=True, text=True, timeout=60)
    with (out / 'setup.stdout.log').open('a') as stdout, (out / 'setup.stderr.log').open('a') as stderr:
        stdout.write(sdk_result.stdout)
        stderr.write(sdk_result.stderr)
    sdk_result.check_returncode()
    if 'Error in sitecustomize' in sdk_result.stderr:
        raise RuntimeError('Python startup failed in sitecustomize; preserve setup.stderr.log in the ZIP.')
    setup_status['stage'] = 'read_secret'
    from kaggle_secrets import UserSecretsClient
    try:
        secret_check = UserSecretsClient().get_secret('GEMINI_API_KEY')
        assert isinstance(secret_check, str) and secret_check
    except Exception:
        raise RuntimeError('Enable the existing GEMINI_API_KEY Secret for this notebook.') from None
    finally:
        secret_check = None
    setup_status.update(state='completed', stage='complete', sdk='2.22.0')
except BaseException as exc:
    setup_status.update(state='failed', error_type=type(exc).__name__)
    if isinstance(exc, subprocess.CalledProcessError): setup_status['exit_code'] = exc.returncode
    raise
finally:
    (out / 'setup-status.json').write_text(json.dumps(setup_status, indent=2))
    bundle_endpoint()
print('SETUP_OK: no live Gemini requests yet. Run cell 2 once.')


In [ ]:
assert 'python' in globals() and Path(python).is_file(), 'Complete cell 1 first'
assert (out / 'setup-status.json').exists() and json.loads((out / 'setup-status.json').read_text()).get('state') == 'completed', 'Cell 1 did not complete; preserve its ZIP and resolve setup first'
assert not (out / 'status.json').exists() and not (out / 'probe').exists(), 'Already attempted; preserve the ZIP and stop'
model = 'gemini-3.6-flash'
status = {'state': 'running', 'model': model}
with (out / 'status.json').open('x') as handle: json.dump(status, handle, indent=2)
child_env = os.environ.copy()
child_env.update(PYTHONPATH=str(code_dir), GEMINI_MODEL=model, GOOGLE_GENAI_USE_VERTEXAI='false')
started = time.monotonic()
try:
    child_env['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
    with (out / 'probe.stdout.log').open('x') as stdout, (out / 'probe.stderr.log').open('x') as stderr:
        process = subprocess.Popen([python, '-u', str(code_dir / 'scripts/kaggle_reader_endpoint_probe.py'),
            '--inputs', str(code_dir / 'inputs.json'), '--output-dir', str(out / 'probe')],
            env=child_env, stdout=stdout, stderr=stderr, start_new_session=True)
        try:
            while True:
                try:
                    code = process.wait(timeout=15)
                    break
                except subprocess.TimeoutExpired:
                    elapsed = time.monotonic() - started
                    print('Endpoint comparison running:', round(elapsed), 'seconds', flush=True)
                    if elapsed >= 600: raise TimeoutError('Endpoint comparison exceeded 600 seconds')
        except BaseException:
            import signal
            os.killpg(process.pid, signal.SIGTERM)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            raise
    status.update(state='completed' if code == 0 else 'failed', exit_code=code)
except BaseException as exc:
    status.update(state='interrupted', error_type=type(exc).__name__)
    raise
finally:
    child_env.pop('GEMINI_API_KEY', None)
    status['elapsed_ms'] = round((time.monotonic() - started) * 1000, 3)
    (out / 'status.json').write_text(json.dumps(status, indent=2))
    bundle_endpoint()
summary = out / 'probe/summary.json'
print(summary.read_text(encoding='utf-8') if summary.exists() else 'No summary; preserve status/stderr in the ZIP.')
print('Stop here. Contract success is not the Reader quality gate; preserve the ZIP for review.')
